# 365 Probabilidades · Dia #078
## Qual a probabilidade de a sua boa notícia testar o relacionamento mais que uma briga?

**Tipo:** Experimental
**Data de publicação:** 2026-08-30
**Ferramenta:** Python
**Decisão analisada:** Onde eu devo olhar para saber como está o meu relacionamento?
**Hashtag:** #365Probabilidades #Dia078

---

### 📖 A História

Quando alguém quer avaliar como está um relacionamento, olha sempre para o mesmo
lugar: a briga.

Como vocês discutem. Se levantam a voz. Se um vira as costas. Se sobra alguma coisa no
dia seguinte. Terapia de casal foi construída em cima disso, e a intuição é boa, porque
o conflito é onde o desgaste aparece de forma visível.

Mas existe uma cena bem mais comum que a briga, e ninguém presta atenção nela.

Alguém chega em casa com uma notícia boa. Passou numa prova, fechou um negócio,
recebeu um elogio, resolveu uma coisa difícil. E conta.

O que acontece nos dez segundos seguintes é o assunto deste dia.

---

### 📚 O Conceito: capitalização

Existe um nome técnico para o ato de compartilhar uma coisa boa com alguém:
**capitalização**.

A ideia é que o evento positivo, sozinho, produz um efeito. E contar esse evento a
alguém produz um efeito **adicional**, que depende inteiramente de como a outra pessoa
responde.

A pesquisa identificou quatro formas de responder, cruzando dois eixos: se a resposta é
**ativa ou passiva**, e se é **construtiva ou destrutiva**.

**Ativa e construtiva:** entusiasmo genuíno, perguntas, quer saber mais.
**Ativa e destrutiva:** esvazia o evento, aponta o problema, lembra do risco.
**Passiva e construtiva:** apoio morno, discreto, sem entusiasmo.
**Passiva e destrutiva:** ignora e muda de assunto.

Repare no terceiro. Ele é gentil, não é grosseiro, não briga. E mesmo assim aparece
associado a desfechos ruins.

Essa é a parte contraintuitiva do dia: **não basta não ser ruim.**

---

### 🧮 O Modelo

Um estudo com interação videogravada, codificação por observadores externos e
seguimento longitudinal.

**Fontes:**
- Gable, S. L., Gonzaga, G. C. & Strachman, A., 2006 · "Will you be there for me when
  things go right? Supportive responses to positive event disclosures" · *Journal of
  Personality and Social Psychology* 91(5), 904-917 · **79 casais em namoro** ·
  medidas de bem-estar da relação, seguidas de **interações videogravadas** em que os
  dois se revezavam discutindo eventos positivos e negativos recentes · quem contava
  avaliava o quanto se sentiu compreendido, validado e cuidado · **observadores
  externos codificaram o comportamento de quem respondia**
- **Achado central:** dois meses depois, tanto os autorrelatos quanto os códigos dos
  observadores mostraram que **as respostas às discussões de eventos positivos estavam
  mais relacionadas ao bem-estar da relação e ao término do que as respostas às
  discussões de eventos negativos**
- Gable, S. L., Reis, H. T., Impett, E. A. & Asher, E. R., 2004 · *JPSP* 87(2),
  228-245 · quatro estudos · comunicar eventos positivos se associou a mais afeto
  positivo e bem-estar diário, **acima e além do impacto do próprio evento** e de
  outros eventos do dia · quando o outro respondia de forma ativa e construtiva, e não
  passiva ou destrutiva, os benefícios eram maiores · relações em que o parceiro
  responde com entusiasmo se associaram a mais intimidade e mais satisfação conjugal
  diária

**Pendência declarada:** o N dos quatro estudos de 2004 não foi localizado nas fontes
acessíveis. O N de 2006, que é o estudo central deste dia, está publicado: 79 casais.

**Nota metodológica sobre o fator ×0.80:** não se aplica. Interação videogravada com
codificação por observadores externos, mais medidas repetidas com seguimento de dois
meses.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

Bibliotecas carregadas


In [2]:
# --- DADOS DA LITERATURA ---
# Gable, Gonzaga & Strachman, 2006, JPSP 91(5), 904-917

n_casais_2006   = 79
seguimento_meses = 2
metodo_2006 = ['medidas de bem-estar da relacao',
               'interacao videogravada sobre evento POSITIVO',
               'interacao videogravada sobre evento NEGATIVO',
               'autoavaliacao de sentir-se compreendido, validado e cuidado',
               'codificacao por observadores externos',
               'reavaliacao dois meses depois']

# Desfechos avaliados no seguimento
desfechos_2006 = ['bem-estar do relacionamento', 'termino do relacionamento']

# Achado central: qual conversa previu melhor os desfechos
PREDITOR_MAIS_FORTE = 'respostas a eventos POSITIVOS'
CONFIRMADO_POR = ['autorrelato', 'codificacao observacional']

# Gable, Reis, Impett & Asher, 2004 - as quatro formas de responder
# (eixo ativo/passivo x construtivo/destrutivo) e a direcao do desfecho
RESPOSTAS = {
    'Ativa e construtiva':  {'eixo': (1, 1),   'desfecho': 'positivo',
                             'exemplo': 'entusiasmo, perguntas, quer saber mais'},
    'Ativa e destrutiva':   {'eixo': (1, -1),  'desfecho': 'negativo',
                             'exemplo': 'esvazia o evento, aponta o risco'},
    'Passiva e construtiva': {'eixo': (-1, 1), 'desfecho': 'negativo',
                              'exemplo': 'apoio morno, discreto, sem entusiasmo'},
    'Passiva e destrutiva': {'eixo': (-1, -1), 'desfecho': 'negativo',
                             'exemplo': 'ignora e muda de assunto'},
}

n_2004 = None   # NAO LOCALIZADO nas fontes acessiveis. Nao inventar.

aplica_fator_080 = False

print("=" * 72)
print("  DADOS - O QUE ACONTECE QUANDO VOCE CONTA UMA COISA BOA")
print("=" * 72)
print(f"\n  Gable, Gonzaga & Strachman, 2006 (JPSP 91(5), 904-917):")
print(f"  -> {n_casais_2006} casais em namoro")
print(f"  -> Seguimento de {seguimento_meses} meses")
print(f"\n  O que o estudo fez:")
for etapa in metodo_2006:
    print(f"  -> {etapa}")
print(f"\n  Desfechos avaliados no seguimento:")
for d in desfechos_2006:
    print(f"  -> {d}")
print(f"\n  ACHADO CENTRAL:")
print(f"  -> O que previu melhor os dois desfechos foram as")
print(f"     {PREDITOR_MAIS_FORTE}, e nao as respostas a eventos negativos.")
print(f"  -> Confirmado por: {', '.join(CONFIRMADO_POR)}")
print(f"\n  As quatro formas de responder (Gable et al., 2004):")
print(f"  {'forma':<24} {'desfecho':<10} exemplo")
print(f"  {'-'*24} {'-'*10} {'-'*44}")
for nome, v in RESPOSTAS.items():
    print(f"  {nome:<24} {v['desfecho']:<10} {v['exemplo']}")
print(f"\n  -> N dos estudos de 2004: NAO LOCALIZADO")
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 72)

  DADOS - O QUE ACONTECE QUANDO VOCE CONTA UMA COISA BOA

  Gable, Gonzaga & Strachman, 2006 (JPSP 91(5), 904-917):
  -> 79 casais em namoro
  -> Seguimento de 2 meses

  O que o estudo fez:
  -> medidas de bem-estar da relacao
  -> interacao videogravada sobre evento POSITIVO
  -> interacao videogravada sobre evento NEGATIVO
  -> autoavaliacao de sentir-se compreendido, validado e cuidado
  -> codificacao por observadores externos
  -> reavaliacao dois meses depois

  Desfechos avaliados no seguimento:
  -> bem-estar do relacionamento
  -> termino do relacionamento

  ACHADO CENTRAL:
  -> O que previu melhor os dois desfechos foram as
     respostas a eventos POSITIVOS, e nao as respostas a eventos negativos.
  -> Confirmado por: autorrelato, codificacao observacional

  As quatro formas de responder (Gable et al., 2004):
  forma                    desfecho   exemplo
  ------------------------ ---------- --------------------------------------------
  Ativa e construtiva      positivo 

In [3]:
# --- O MODELO ---

# 1) Quantas formas de responder levam a desfecho positivo?
positivas = [n for n, v in RESPOSTAS.items() if v['desfecho'] == 'positivo']
negativas = [n for n, v in RESPOSTAS.items() if v['desfecho'] == 'negativo']

# 2) E quantas delas sao "gentis", ou seja, construtivas?
construtivas = [n for n, v in RESPOSTAS.items() if v['eixo'][1] == 1]
construtivas_que_falham = [n for n in construtivas if RESPOSTAS[n]['desfecho'] == 'negativo']

# 3) O que 79 casais conseguem enxergar.
#    O achado principal e uma COMPARACAO entre dois preditores, o que exige
#    ainda mais precisao. Assinatura: efeito minimo detectavel em correlacao.
ALFA, PODER = 0.05, 0.80


def poder_correlacao(r, n, alfa=ALFA):
    if r <= 0:
        return alfa
    z = 0.5 * np.log((1 + r) / (1 - r))
    ep = 1 / np.sqrt(n - 3)
    critico = stats.norm.ppf(1 - alfa / 2)
    p = stats.norm.sf(critico - abs(z) / ep) + stats.norm.cdf(-critico - abs(z) / ep)
    return float(np.nan_to_num(p, nan=0.0))


r_minimo = optimize.brentq(
    lambda r: poder_correlacao(r, n_casais_2006, ALFA) - PODER, 1e-4, 0.99)

convencionais = {'pequena (r=0,10)': 0.10, 'moderada (r=0,30)': 0.30,
                 'forte (r=0,50)': 0.50}

print("=" * 72)
print("  MODELO - A MATEMATICA SIMPLES E A RESSALVA HONESTA")
print("=" * 72)
print(f"\n  1) Das {len(RESPOSTAS)} formas de responder a uma boa noticia,")
print(f"     apenas {len(positivas)} leva a desfecho positivo: {positivas[0]}.")
print(f"  -> As outras {len(negativas)} aparecem associadas a desfecho negativo.")
print(f"\n  2) E a parte contraintuitiva:")
print(f"  -> {len(construtivas)} das 4 respostas sao CONSTRUTIVAS, ou seja, gentis.")
print(f"  -> Mas {len(construtivas_que_falham)} delas ainda assim falha:"
      f" {construtivas_que_falham[0]}.")
print(f"  -> Nao basta nao ser ruim. Apoio morno fica do lado errado da tabela.")
print(f"\n  3) O que {n_casais_2006} casais conseguem enxergar:")
print(f"  -> Efeito minimo detectavel, com {PODER*100:.0f}% de poder:"
      f" r = {r_minimo:.2f}")
print(f"  -> Poder por tamanho de correlacao:")
for nome, r in convencionais.items():
    print(f"     {nome:<20} poder = {poder_correlacao(r, n_casais_2006)*100:.0f}%")
print(f"  -> Ou seja: o estudo enxerga associacao moderada para cima.")
print(f"     Uma diferenca pequena entre os dois tipos de conversa")
print(f"     poderia ter passado despercebida.")
print(f"\n  4) O que sustenta a leitura mesmo assim:")
print(f"  -> O achado aparece em DUAS fontes de dado independentes:")
print(f"     o autorrelato de quem contou e a codificacao de observadores externos.")
print(f"  -> E aparece em DOIS desfechos: bem-estar e termino.")
print(f"  -> Convergencia entre metodos vale mais que tamanho de amostra sozinho.")
print("=" * 72)

  MODELO - A MATEMATICA SIMPLES E A RESSALVA HONESTA

  1) Das 4 formas de responder a uma boa noticia,
     apenas 1 leva a desfecho positivo: Ativa e construtiva.
  -> As outras 3 aparecem associadas a desfecho negativo.

  2) E a parte contraintuitiva:
  -> 2 das 4 respostas sao CONSTRUTIVAS, ou seja, gentis.
  -> Mas 1 delas ainda assim falha: Passiva e construtiva.
  -> Nao basta nao ser ruim. Apoio morno fica do lado errado da tabela.

  3) O que 79 casais conseguem enxergar:
  -> Efeito minimo detectavel, com 80% de poder: r = 0.31
  -> Poder por tamanho de correlacao:
     pequena (r=0,10)     poder = 14%
     moderada (r=0,30)    poder = 77%
     forte (r=0,50)       poder = 100%
  -> Ou seja: o estudo enxerga associacao moderada para cima.
     Uma diferenca pequena entre os dois tipos de conversa
     poderia ter passado despercebida.

  4) O que sustenta a leitura mesmo assim:
  -> O achado aparece em DUAS fontes de dado independentes:
     o autorrelato de quem contou e a 

In [4]:
# --- VISUALIZACAO ---

DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - A matriz das quatro respostas
fig1, ax1 = plt.subplots(figsize=(12, 9))

ax1.axhline(y=0, color='#333', linewidth=1.8)
ax1.axvline(x=0, color='#333', linewidth=1.8)

for nome, v in RESPOSTAS.items():
    x, y = v['eixo'][1], v['eixo'][0]      # x = construtivo, y = ativo
    cor = VERDE if v['desfecho'] == 'positivo' else VERMELHO
    alpha = 0.16 if v['desfecho'] == 'positivo' else 0.09
    ax1.add_patch(plt.Rectangle((0 if x > 0 else -1, 0 if y > 0 else -1), 1, 1,
                                facecolor=cor, alpha=alpha, edgecolor=cor,
                                linewidth=2))
    ax1.text(x * 0.5, y * 0.5 + 0.22, nome, ha='center', va='center', fontsize=15,
             fontweight='bold', color=cor)
    ax1.text(x * 0.5, y * 0.5 - 0.06, v['exemplo'], ha='center', va='center',
             fontsize=11, color='#333', wrap=True)
    marca = 'desfecho positivo' if v['desfecho'] == 'positivo' else 'desfecho negativo'
    ax1.text(x * 0.5, y * 0.5 - 0.30, marca, ha='center', va='center', fontsize=11,
             style='italic', color=cor)

ax1.text(0, 1.12, 'ATIVA', ha='center', fontsize=13, fontweight='bold', color=CINZA)
ax1.text(0, -1.16, 'PASSIVA', ha='center', fontsize=13, fontweight='bold', color=CINZA)
ax1.text(1.14, 0, 'CONSTRUTIVA', va='center', fontsize=13, fontweight='bold',
         color=CINZA, rotation=270)
ax1.text(-1.14, 0, 'DESTRUTIVA', va='center', fontsize=13, fontweight='bold',
         color=CINZA, rotation=90)

ax1.set_xlim(-1.35, 1.35)
ax1.set_ylim(-1.35, 1.35)
ax1.set_xticks([]); ax1.set_yticks([])
ax1.grid(False)
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.set_title('As quatro formas de responder a uma boa notícia\n'
              'Três das quatro levam a desfecho negativo, e uma delas é gentil',
              fontsize=16, pad=20)

plt.figtext(0.5, 0.005,
            'Fonte: Gable, Reis, Impett & Asher, 2004, JPSP 87(2), 228-245'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-078-grafico-01-quatro-respostas.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - O desenho do estudo de 2006
fig2, ax2 = plt.subplots(figsize=(12, 8))
ax2.axis('off')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)

caixa = dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor=CINZA,
             linewidth=1.6)

ax2.text(5, 9.4, f'{n_casais_2006} casais, duas conversas, dois meses depois',
         ha='center', fontsize=17, fontweight='bold')

ax2.text(3.0, 7.7, 'conversa sobre\numa coisa BOA\nque aconteceu',
         ha='center', va='center', fontsize=14,
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#f6f1e2', edgecolor=DOURADO,
                   linewidth=2))
ax2.text(7.0, 7.7, 'conversa sobre\numa coisa RUIM\nque aconteceu',
         ha='center', va='center', fontsize=14, bbox=caixa)

ax2.text(5, 6.0, 'videogravadas e codificadas por observadores externos',
         ha='center', fontsize=12, color=CINZA, style='italic')

for x in (3.0, 7.0):
    ax2.annotate('', xy=(x, 4.6), xytext=(x, 6.6),
                 arrowprops=dict(arrowstyle='->', color=CINZA, lw=2))

ax2.text(5, 3.9,
         'Dois meses depois: bem-estar da relação e término',
         ha='center', fontsize=14, fontweight='bold', color='#333')

ax2.text(5, 2.4,
         'A conversa sobre a coisa BOA previu melhor os dois desfechos.',
         ha='center', fontsize=16, fontweight='bold', color=DOURADO,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#f6f1e2', edgecolor=DOURADO,
                   linewidth=2))
ax2.text(5, 0.9,
         'O mesmo resultado apareceu no autorrelato de quem contou e na codificação\n'
         'independente dos observadores.',
         ha='center', fontsize=12, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Gable, Gonzaga & Strachman, 2006, JPSP 91(5), 904-917'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-078-grafico-02-desenho.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - Assinatura: o que 79 casais conseguem enxergar
fig3, ax3 = plt.subplots(figsize=(12, 8))

rs = np.linspace(0.02, 0.7, 400)
ax3.plot(rs, [poder_correlacao(r, n_casais_2006) * 100 for r in rs],
         color=DOURADO, linewidth=3.2)
ax3.axhline(y=80, color=CINZA, linestyle='--', linewidth=1.8)
ax3.axvline(x=r_minimo, color=DOURADO, linestyle=':', linewidth=2.4)

for nome, r in convencionais.items():
    p = poder_correlacao(r, n_casais_2006) * 100
    ax3.scatter([r], [p], s=140, color=VERDE, zorder=3)
    ax3.text(r, p + 4, f'{nome.split(" ")[0]}\n{p:.0f}%', ha='center', fontsize=11,
             color=VERDE, fontweight='bold')

ax3.text(r_minimo + 0.012, 22, f'efeito mínimo detectável\nr = {r_minimo:.2f}'
         .replace('.', ','), fontsize=13, color=DOURADO, fontweight='bold')
ax3.text(0.69, 82, 'poder de 80%', ha='right', fontsize=11, color=CINZA)

ax3.set_xlim(0, 0.7)
ax3.set_ylim(0, 108)
ax3.set_xlabel('Correlação entre a resposta do parceiro e o desfecho da relação')
ax3.set_ylabel('Chance de o estudo enxergar essa correlação (%)')
ax3.set_title(f'A assinatura estatística: o que {n_casais_2006} casais conseguem ver\n'
              'O estudo enxerga associação moderada para cima, não diferenças pequenas',
              fontsize=15, pad=18)
ax3.text(0.5, -0.135,
         'O que sustenta a leitura não é o tamanho da amostra: é a convergência entre '
         'autorrelato e\ncodificação independente de observadores, em dois desfechos '
         'diferentes.',
         transform=ax3.transAxes, ha='center', fontsize=11, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Cálculo de poder por transformação z de Fisher, alfa = 0,05'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-078-grafico-03-poder.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

Grafico 1 salvo
Grafico 2 salvo
Grafico 3 salvo


### 💡 O Insight

Setenta e nove casais. Duas conversas videogravadas, uma sobre uma coisa boa que
aconteceu e outra sobre uma coisa ruim. Observadores externos codificando o
comportamento de quem escutava. E uma reavaliação dois meses depois.

**A conversa sobre a coisa boa previu melhor o futuro do casal do que a conversa sobre
a coisa ruim.** Nos dois desfechos: bem-estar da relação e término.

E não é só o que as pessoas disseram sobre si. O mesmo resultado apareceu na
codificação independente de quem assistiu aos vídeos.

Isso contraria a intuição inteira. A gente avalia relacionamento pela briga. Terapia
de casal foi construída em cima do conflito. E o que aparece como melhor previsor é a
cena mais banal que existe: alguém chega com uma notícia boa e conta.

A parte que eu acho mais dura vem do mapa das quatro respostas.

Das quatro formas de reagir a uma boa notícia, **só uma leva a desfecho positivo**: o
entusiasmo genuíno, com perguntas, querendo saber mais.

Duas são obviamente ruins: esvaziar o evento e ignorar.

E a quarta é a que me pegou. **Apoio morno.** "Que bom, amor", dito sem levantar a
cabeça. Ela é gentil, não é grosseira, ninguém brigou. E aparece do lado errado da
tabela, junto com as destrutivas.

Não basta não ser ruim.

Existe uma razão para isso fazer sentido, e ela é quase óbvia depois de dita. Quando
você chega mal, quem te escuta está cumprindo uma obrigação social clara: consolar é o
que se espera. Quando você chega bem, não há obrigação nenhuma. Ninguém é cobrado por
comemorar pouco.

Por isso a alegria é o teste mais duro. Ela não vem com roteiro.

E se você acha que está do lado certo, vale uma pergunta honesta: quando foi a última
vez que você fez uma **pergunta** sobre a boa notícia de alguém, em vez de só dizer que
era ótima?

*Qual foi a última boa notícia que te contaram, e o que você fez com ela?*

---

### ⚠️ Limitações do Modelo

- **São 79 casais.** É amostra pequena para um achado que é uma **comparação entre dois
  preditores**. Com esse N, o estudo enxerga associação moderada para cima; uma
  diferença pequena entre os dois tipos de conversa poderia ter passado despercebida.
- **Associação, não causa.** Ninguém sorteou como cada parceiro responderia. Casais que
  já vão bem podem simplesmente comemorar mais, e aí a seta aponta para trás.
- **Casais em namoro, não casados**, e o seguimento é de dois meses. Nada aqui foi
  testado em relações longas nem em prazos de anos.
- **O N dos quatro estudos de 2004 não foi localizado** nas fontes acessíveis. O
  estudo central deste dia, o de 2006, tem N publicado.
- **A classificação em quatro respostas é uma simplificação.** Ninguém responde sempre
  do mesmo jeito, e a mesma frase muda de categoria conforme tom, contexto e história
  do casal.
- **Interação em laboratório, com câmera.** Saber que está sendo gravado muda
  comportamento, e isso tende a empurrar todo mundo para a versão mais apresentável de
  si.
- **O que sustenta a leitura não é o tamanho da amostra**, e sim a convergência entre
  duas fontes independentes de dado, autorrelato e observação externa, em dois
  desfechos diferentes.
- Fator ×0.80 não aplicado: interação videogravada com codificação por observadores.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
